Welcome to spinqick!

In [4]:
import Pyro4
from qick import QickConfig
from spinqick.models import hardware_config_models, dcs_model
import pathlib

%matplotlib widget

First we can connect to our qick board.  We use a pyro server to communicate with our board, which allows us to run python code on our computer instead of on the small cpu provided on the board.

In [2]:
Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

ns_host = "<your-board-ip-address>"  # make sure this matches your board's ip address!
ns_port = "<your-nameserver-port>"
proxy_name = "myqick"

ns = Pyro4.locateNS(host=ns_host, port=ns_port)
soc = Pyro4.Proxy(ns.lookup(proxy_name))
soccfg = QickConfig(soc.get_cfg())
print(soccfg)

QICK running on ZCU216, software version 0.2.360

Firmware configuration (built Fri Aug  1 10:30:05 2025):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 2, DAC tile 3], [ADC tile 1, ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=6881.280 Msps, fabric=430.080 MHz
		envelope memory: 65536 complex samples (9.524 us)
		32-bit DDS, range=6881.280 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_signal_gen_v6 - fs=6881.280 Msps, fabric=430.080 MHz
		envelope memory: 65536 complex samples (9.524 us)
		32-bit DDS, range=6881.280 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_signal_gen_v6 - fs=6881.280 Msps, fabric=430.080 MHz
		envelope memory: 65536 real samples (9.524 us)
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis_signal_gen_v6 - fs=6881.280 Msps, fabric=430.080 MHz
		envelope m

Dot experiments can be complex and require us to store and organize many parameters for every experiment.  Most of the parameters used in spinqick are passed into the experiment code using a dictionary which we call a config. Some parameters in the config are shared across many experiments, for example the setup of readout parameters like source-drain chop amplitude. We store these parameters in dictionaries which are saved to yaml files, with filepaths specified in spinqick.settings.filesettings.

In spinqick.models we've made templates for hardware and readout config files, so we can use those to create some example configs now.  We start with the hardware config, which is a hardware map describing all of the low-speed and high-speed lines connecting to each gate.


In [6]:
gate_b = hardware_config_models.SlowGate(
    dc_conversion_factor=4,
    gate_type="plunger",
    slow_dac_address="slow_dac_address",
    slow_dac_channel=1,
    crosscoupling=None,
    max_v=1,
)
person_dict = gate_b.model_dump()
gate_a = hardware_config_models.FastGate(
    **person_dict, dac_conversion_factor=1, qick_gen=0
)
gate_a

FastGate(dc_conversion_factor=4.0, slow_dac_address='slow_dac_address', slow_dac_channel=1, max_v=1.0, gate_type=<GateTypes.PLUNGER: 'plunger'>, crosscoupling=None, dac_conversion_factor=1.0, qick_gen=0)

In [ ]:
json_string = pathlib.Path("hw_cfg_example.json").read_text()
person = hardware_config_models.HardwareConfig.model_validate_json(json_string)
person

HardwareConfig(sd_in=SourceDrainIn(qick_gen=0), sd_out=SourceDrainOut(qick_adc=0), rf_gen=None, slow_dac_trig_pin=0, rf_trig_pin=None, channels={'A': FastGate(dc_conversion_factor=4.0, slow_dac_address='slow_dac_address', slow_dac_channel=0, max_v=1.0, gate_type='P', crosscoupling=None, dac_conversion_factor=10000.0, qick_gen=1), 'B': SlowGate(dc_conversion_factor=4.0, slow_dac_address='slow_dac_address', slow_dac_channel=1, max_v=1.0, gate_type='P', crosscoupling=None)}, voltage_source='test')

Next we store information in the readout config dictionary. This dictionary holds some frequently used readout parameters, and every spinqick function uses the readout config. To get started we only need the dcs_cfg part.

In [5]:
readout = dcs_model.DcsConfigParams(
    adc_trig_offset=0,
    dds_freq=1,
    readout_freq=1,
    length=1,
    readout_length=1,
    pulse_gain_readout=0.9,
    slack_delay=0,
)

readout_cfg = readout.model_dump()
readout_cfg

{'adc_trig_offset': 0.0,
 'dds_freq': 1.0,
 'readout_freq': 1.0,
 'length': 1.0,
 'readout_length': 1.0,
 'pulse_gain_readout': 0.9,
 'slack_delay': 0.0,
 'ac_gate_gain': None}

Now we're ready to try running an experiment.  We will load a simple experiment from the system_calibrations module. This experiment just sweeps a single parameter called 'adc_trig_offset' which is the delay between when the dac outputs a pulse and when the adc begins to collect data.

TODO fill in the rest of this notebook!
